<a href="https://colab.research.google.com/github/kjahan/self_taught_reasoner/blob/main/notebooks/verifiers/cot_reasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chain of Thought

CoT helps LMs with their reasoning for  for arithmetic, commonsense, and symbolic reasoning benchmarks!

https://arxiv.org/pdf/2201.11903.pdf

Chain-of-thought prompting has several attractive properties as an approach for facilitating reasoning in language models.


`1. First, chain of thought, in principle, allows models to decompose multi-step problems into intermediate steps, which means that additional computation can be allocated to problems that require more reasoning steps.`

`2. Second, a chain of thought provides an interpretable window into the behavior of the model, suggesting how it might have arrived at a particular answer and providing opportunities to debug where the reasoning path went wrong (although fully characterizing a model’s computations that support an answer remains an open question).`

`3. Third, chain-of-thought reasoning can be used for tasks such as math word problems, commonsense reasoning, and symbolic manipulation, and is potentially applicable (at least in principle) to any task that humans can solve via language.`

`4. Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting.`

In [1]:
!pip3 install --upgrade openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.8/220.8 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 12.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 12.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llmx 0.0.15a0 requires cohere, which is not installed.
llmx 0.0.15a0 requires tiktoken, which is not installed.


In [2]:
import openai
import getpass

from io import StringIO
from contextlib import redirect_stdout

In [3]:
try:
    openai.api_key = getpass.getpass()
except Exception as error:
    print('ERROR', error)

··········


In [4]:
MODEL_3 = "gpt-3.5-turbo"
MODEL_4 = "gpt-4" #"gpt-4-0613"

## Test Sets

In [22]:
q1 = "Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"

a1 = "A: Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11."

## Target test

GSM8k dataset


`James writes a 3-page letter to 2 different friends twice a week.  How many pages does he write a year?`

`He writes each friend 3*2=<<3*2=6>>6 pages a week`

`So he writes 6*2=<<6*2=12>>12 pages every week`

`That means he writes 12*52=<<12*52=624>>624 pages a year`

`#### 624`


-----------------------------------
`Tina makes $18.00 an hour.  If she works more than 8 hours per shift, she is eligible for overtime, which is paid by your hourly wage + 1/2 your hourly wage.  If she works 10 hours every day for 5 days, how much money does she make?`

`She works 8 hours a day for $18 per hour so she makes 8*18 = $<<8*18=144.00>>144.00 per 8-hour shift
She works 10 hours a day and anything over 8 hours is eligible for overtime, so she gets 10-8 = <<10-8=2>>2 hours of overtime`

`Overtime is calculated as time and a half so and she makes $18/hour so her overtime pay is 18*.5 = $<<18*.5=9.00>>9.00`

`Her overtime pay is 18+9 = $<<18+9=27.00>>27.00`

`Her base pay is $144.00 per 8-hour shift and she works 5 days and makes 5 * $144 = $<<144*5=720.00>>720.00`

`Her overtime pay is $27.00 per hour and she works 2 hours of overtime per day and makes 27*2 = $<<27*2=54.00>>54.00 in overtime pay
2 hours of overtime pay for 5 days means she makes 54*5 = $270.00`

`In 5 days her base pay is $720.00 and she makes $270.00 in overtime pay so she makes $720 + $270 = $<<720+270=990.00>>990.00`

`#### 990`

In [19]:
test_q1 = "James writes a 3-page letter to 2 different friends twice a week.  How many pages does he write a year?"
test_q2 = "Anna writes a  6-page letter to 3 different friends four times a week.  How many pages does he write a year?"
test_q3 = "Tina makes $18.00 an hour.  If she works more than 8 hours per shift, she is eligible for overtime, which is paid by your hourly wage + 1/2 your hourly wage.  If she works 10 hours every day for 5 days, how much money does she make?"
test_q4 = "Q: A coin is heads down. Maybelle flips the coin. Shalonda does not flip the coin. Is the coin still heads up?"
test_q5 = "Q: A coin is heads up. Maybelle flips the coin. Shalonda does not flip the coin. Is the coin still heads up?"

## Test WO COT!

In [8]:
system_prompt = ""
user_prompt = test_q1

completion = openai.chat.completions.create(
  model=MODEL_3,
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
  ]
)

print(completion.choices[0].message.content)

James writes a total of 2 friends x 3 pages = <<2*3=6>>6 pages per week.
James writes a total of 6 pages x 2 times per week = <<6*2=12>>12 pages per week.
Over a year he writes a total of 12 pages per week x 52 weeks = <<12*52=624>>624 pages. Answer: \boxed{624}.


In [12]:
system_prompt = ""
user_prompt = test_q2

completion = openai.chat.completions.create(
  model=MODEL_3,
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
  ]
)

print(completion.choices[0].message.content)

Anna writes a letter to 3 friends four times a week, so she writes 3*4=<<3*4=12>>12 letters a week.
Each letter is 6 pages long, so she writes 12*6=<<12*6=72>>72 pages a week.
There are 52 weeks in a year, so Anna writes 72*52=<<72*52=3744>>3744 pages a year. Answer: \boxed{3744}.


In [13]:
system_prompt = ""
user_prompt = test_q3

completion = openai.chat.completions.create(
  model=MODEL_3,
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
  ]
)

print(completion.choices[0].message.content)

Tina earns $18.00 an hour, so her overtime rate is 1/2 * $18.00 = $<<18*1/2=9.00>>9.00 per hour.
Tina works 8 hours a day, which is her regular working hours per day, and earns 8 * $18.00 = $<<8*18=144.00>>144.00.
She works 10 hours a day for 5 days, which results in 10 - 8 = <<10-8=2>>2 hours of overtime each day.
Her overtime pay per day is 2 * $9.00 = $<<2*9=18.00>>18.00.
Her total overtime pay for the week is 5 * $18.00 = $<<5*18=90.00>>90.00.
Her total regular pay for the week is 5 * $144.00 = $<<5*144=720.00>>720.00.
Her total pay for the week is $90.00 + $720.00 = $<<90+720=810.00>>810.00. Answer: \boxed{810}.


## It seems it has memorized above examples!

Lets try one example from the paper:
https://arxiv.org/pdf/2201.11903.pdf

Figure 3 --> `modified`

`test_q4 = "Q: A coin is heads down. Maybelle flips the coin. Shalonda does not flip the coin. Is the coin still heads up?"`

In [17]:
system_prompt = ""
user_prompt = test_q4

completion = openai.chat.completions.create(
  model=MODEL_3,
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
  ]
)

print(completion.choices[0].message.content)

A: No, if Maybelle flips the coin, the result of the flip will determine whether it is still heads down or if it becomes heads up.


In [18]:
system_prompt = ""
user_prompt = test_q4

completion = openai.chat.completions.create(
  model=MODEL_4,
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
  ]
)

print(completion.choices[0].message.content)

The text does not provide enough information to determine the position of the coin after Maybelle flips it.


In [20]:
system_prompt = ""
user_prompt = test_q5

completion = openai.chat.completions.create(
  model=MODEL_4,
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
  ]
)

print(completion.choices[0].message.content)

A: The coin's position after Maybelle flips it is uncertain. It could be either heads up or tails up.


## Clearly struggles!

# Let's do multi-shot with COT

In [23]:
system_prompt = q1 + "\n" + a1
user_prompt = test_q4

In [24]:
system_prompt

'Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?\nA: Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11.'

In [25]:
user_prompt

'Q: A coin is heads down. Maybelle flips the coin. Shalonda does not flip the coin. Is the coin still heads up?'

In [26]:
system_prompt = ""
user_prompt = test_q4

completion = openai.chat.completions.create(
  model=MODEL_3,
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
  ]
)

print(completion.choices[0].message.content)

No, the coin is no longer heads down but heads up after Maybelle flips it.


In [27]:
system_prompt = ""
user_prompt = test_q4

completion = openai.chat.completions.create(
  model=MODEL_4,
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
  ]
)

print(completion.choices[0].message.content)

The text does not provide information on the result of Maybelle's coin flip.
